# 03 — DFT Fundamentals: From Hohenberg-Kohn to Modern Functionals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/03_dft_fundamentals.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- State and explain the two Hohenberg-Kohn theorems
- Describe the Kohn-Sham DFT formalism and its connection to HF
- Classify exchange-correlation functionals on Jacob's Ladder
- Compare LDA, GGA, meta-GGA, hybrid, and double-hybrid functionals
- Understand dispersion corrections (DFT-D3, D4)
- Choose an appropriate functional for a given chemical problem

## 1. Theory: From Density to Energy

### 1.1 Hohenberg-Kohn Theorems (1964)

DFT is built on two exact theorems by Hohenberg and Kohn:

**Theorem 1 (Existence):** The external potential $v_{ext}(\mathbf{r})$ — and hence the
total energy — is a unique functional of the ground-state electron density $n_0(\mathbf{r})$:

$$E = E[n_0]$$

**Theorem 2 (Variational principle):** The universal functional $F[n]$ delivers
the ground-state energy if and only if the true ground-state density is used:

$$E[n] \geq E_0 \quad \text{for all } n \neq n_0$$

These theorems establish that $n(\mathbf{r})$ — a function of only 3 spatial variables
— contains all information needed to determine the ground-state energy.

### 1.2 Kohn-Sham DFT (1965)

The practical implementation by Kohn and Sham introduces a fictitious non-interacting
reference system with the **same density** as the real interacting system.

The Kohn-Sham equations are:

$$\left[-\frac{1}{2}\nabla^2 + v_{eff}(\mathbf{r})\right]\phi_i(\mathbf{r}) = \epsilon_i \phi_i(\mathbf{r})$$

where the effective potential is:

$$v_{eff}(\mathbf{r}) = v_{ext}(\mathbf{r}) + \underbrace{\int \frac{n(\mathbf{r}')}{|\mathbf{r}-\mathbf{r}'|} d\mathbf{r}'}_{v_{Hartree}} + \underbrace{v_{xc}[n](\mathbf{r})}_{exchange-correlation}$$

The total DFT energy is:

$$E_{KS}[n] = T_s[n] + E_{ne}[n] + J[n] + E_{xc}[n]$$

where $T_s$ = kinetic energy of non-interacting electrons, $E_{ne}$ = nuclear-electron
attraction, $J$ = classical Coulomb (Hartree) energy, and $E_{xc}$ = exchange-correlation energy.

### 1.3 The Exchange-Correlation Functional

The exact $E_{xc}[n]$ is unknown. All DFT methods differ in how they approximate it.

$$E_{xc}[n] = E_x[n] + E_c[n]$$

**Exchange energy** $E_x$ accounts for the Fermi hole (Pauli antisymmetry),
**correlation energy** $E_c$ captures dynamic and static electron correlation.

### 1.4 Dispersion Interactions

Standard DFT functionals miss long-range London dispersion ($r^{-6}$ decay).
The DFT-D3 correction by Grimme adds:

$$E_{DFT-D3} = -\frac{1}{2}\sum_{i\neq j} s_6 \frac{C_6^{ij}}{r_{ij}^6} f_{damp}(r_{ij}) + s_8 \frac{C_8^{ij}}{r_{ij}^8} f_{damp}(r_{ij})$$

Use keywords in ORCA: `B3LYP-D3BJ/def2-SVP` or `PBE0-D3/def2-TZVP`

In [ ]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 03: DFT Fundamentals
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from pyscf import gto, dft, scf

# ------------------------------------------------------------------
# Jacob's Ladder of DFT Approximations
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis('off')

# Color gradient from bottom (warm) to top (cool)
rung_colors = ['#F9A825', '#FB8C00', '#E64A19', '#8E24AA', '#1565C0']
rung_data = [
    (1, 'Rung 1: LDA',      'Local Density\nApproximation',
     'PW92, VWN, SVWN',      r'$n(\mathbf{r})$'),
    (2, 'Rung 2: GGA',       'Generalized Gradient\nApproximation',
     'PBE, BLYP, BP86',       r'$n, \nabla n$'),
    (3, 'Rung 3: meta-GGA',  'Kinetic Energy Density',
     'TPSS, M06-L, SCAN',     r'$n, \nabla n, \tau$'),
    (4, 'Rung 4: Hybrid',    'Exact HF Exchange\n(partial)',
     'B3LYP, PBE0, M06',      r'$n, \nabla n, E_x^{HF}$'),
    (5, 'Rung 5: Dbl Hybrid','HF Exchange + MP2\nCorrelation',
     r'$\omega$B97X-D, B2-PLYP',r'$n, \nabla n, E_x^{HF}, E_c^{MP2}$'),
]

# Draw ladder poles
ax.plot([1.5, 1.5], [0.3, 10.2], 'k-', lw=3, zorder=1)
ax.plot([8.5, 8.5], [0.3, 10.2], 'k-', lw=3, zorder=1)

for (rung_num, title, subtitle, functionals, ingredients) in rung_data:
    y_center = rung_num * 1.75 - 0.8
    color = rung_colors[rung_num - 1]
    
    # Rung (horizontal bar)
    rung_patch = mpatches.FancyBboxPatch((1.6, y_center - 0.5), 6.8, 1.0,
                                          boxstyle="round,pad=0.05",
                                          facecolor=color, edgecolor='white',
                                          linewidth=2, alpha=0.88, zorder=2)
    ax.add_patch(rung_patch)
    
    # Title text
    ax.text(5.0, y_center + 0.15, title, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white', zorder=3)
    
    # Functional examples (left)
    ax.text(2.2, y_center - 0.22, functionals, ha='left', va='center',
            fontsize=8.5, color='white', zorder=3, style='italic')
    
    # Ingredients (right)
    ax.text(7.8, y_center, ingredients, ha='right', va='center',
            fontsize=8.5, color='white', zorder=3)

# 'Chemical Accuracy' bar at top
ca_patch = mpatches.FancyBboxPatch((1.0, 9.2), 8.0, 0.8,
                                    boxstyle="round,pad=0.1",
                                    facecolor='#2ECC71', edgecolor='gold',
                                    linewidth=3, alpha=0.9, zorder=2)
ax.add_patch(ca_patch)
ax.text(5.0, 9.62, '🎯 Chemical Accuracy: < 1 kcal/mol',
        ha='center', va='center', fontsize=12, fontweight='bold',
        color='white', zorder=3)

# 'Heaven' / exact label
ax.text(5.0, 10.6, '☁  Heaven: Exact Exchange-Correlation',
        ha='center', va='center', fontsize=10, color='gray', style='italic')

# Ground label
ax.text(5.0, 0.1, '⚡ Hartree World (no XC)',
        ha='center', va='center', fontsize=9.5, color='#7F8C8D')

# Title
ax.set_title("Jacob's Ladder of Density Functional Approximations",
             fontsize=14, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# DFT Functional Comparison on H₂O / def2-SVP
# ------------------------------------------------------------------

mol = gto.Mole()
mol.atom = '''O  0.000000  0.000000  0.117176
H  0.000000  0.757001 -0.468704
H  0.000000 -0.757001 -0.468704'''
mol.basis = 'def2-SVP'
mol.verbose = 0
mol.build()

functionals = {
    'SVWN':  ('LDA',       '#F9A825'),
    'PBE':   ('GGA',       '#FB8C00'),
    'TPSS':  ('meta-GGA',  '#E64A19'),
    'B3LYP': ('Hybrid',    '#8E24AA'),
    'PBE0':  ('Hybrid',    '#1565C0'),
}

results = []
for func, (category, color) in functionals.items():
    mf = dft.RKS(mol)
    mf.xc = func
    mf.verbose = 0
    e = mf.kernel()
    dm = mf.make_rdm1()
    dip = mf.dip_moment(mol, dm, verbose=0)
    dip_mag = np.linalg.norm(dip)
    results.append({
        'Functional': func, 'Category': category,
        'Energy (Ha)': round(e, 6), 'Dipole (D)': round(dip_mag, 3),
        'Color': color
    })
    print(f"  {func:8s} ({category:10s}): E = {e:.6f} Ha  μ = {dip_mag:.3f} D")

df = pd.DataFrame(results)
print("\n")
print(df[['Functional','Category','Energy (Ha)','Dipole (D)']].to_string(index=False))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Total energies
energies = df['Energy (Ha)'].values
funcs = df['Functional'].values
colors_list = df['Color'].values

bars1 = ax1.bar(funcs, energies - energies.max(), color=colors_list,
                edgecolor='white', linewidth=0.5)
ax1.set_xlabel('DFT Functional', fontsize=12)
ax1.set_ylabel('ΔE relative to SVWN (Ha)', fontsize=11)
ax1.set_title('DFT Functional Comparison: H₂O\ndef2-SVP basis', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')
for bar, e in zip(bars1, energies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f'{e:.4f}', ha='center', va='bottom', fontsize=8, rotation=30)

# Panel 2: Dipole moments
dipoles = df['Dipole (D)'].values
bars2 = ax2.bar(funcs, dipoles, color=colors_list, edgecolor='white', linewidth=0.5)
ax2.axhline(y=1.85, color='black', linestyle='--', alpha=0.7, label='Experiment (1.85 D)')
ax2.set_xlabel('DFT Functional', fontsize=12)
ax2.set_ylabel('Dipole Moment (Debye)', fontsize=12)
ax2.set_title('Dipole Moments: H₂O / def2-SVP', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
for bar, d in zip(bars2, dipoles):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{d:.3f}', ha='center', va='bottom', fontsize=9)

# Add category labels
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F9A825', label='LDA'),
    Patch(facecolor='#FB8C00', label='GGA'),
    Patch(facecolor='#E64A19', label='meta-GGA'),
    Patch(facecolor='#8E24AA', label='Hybrid'),
    Patch(facecolor='#1565C0', label='Hybrid'),
]
ax1.legend(handles=legend_elements[:3], fontsize=8, title='Rung')

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# DFT-D3 Dispersion Correction
# ------------------------------------------------------------------
# Standard DFT functionals miss van der Waals / London dispersion
# interactions. Grimme's DFT-D3 correction adds pairwise C6/r^6 terms.

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

# Conceptual: Lennard-Jones-like comparison
r = np.linspace(2.5, 12, 500)   # Angstrom

# Approximate DFT (no dispersion) for benzene dimer: flat near repulsion, no well
def dft_nodispersion(r):
    # Simplified: Pauli repulsion only, very shallow
    return 5.0 * np.exp(-r/1.5) - 0.2 * np.exp(-r/4.0)

# With D3 correction: adds -C6/r^6 term
def dft_d3(r, C6=45.0, s6=1.0):
    return dft_nodispersion(r) - s6 * C6 / r**6

E_nodis = dft_nodispersion(r)
E_d3 = dft_d3(r)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r, E_nodis, '--', color='coral', linewidth=2.5, label='DFT (no dispersion)')
ax.plot(r, E_d3, '-', color='steelblue', linewidth=2.5, label='DFT-D3')
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
ax.fill_between(r, E_d3, 0, where=(E_d3 < 0), alpha=0.15, color='steelblue',
                label='Dispersion well')
ax.set_xlim(2.5, 12)
ax.set_ylim(-0.6, 1.0)
ax.set_xlabel('Intermolecular Distance (Å)', fontsize=12)
ax.set_ylabel('Interaction Energy (kcal/mol)', fontsize=12)
ax.set_title('Effect of DFT-D3 Dispersion Correction\n(Conceptual: benzene dimer)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("DFT-D3 Dispersion Correction Summary:")
print("  ΔE_disp = -Σ_{i<j} s₆ C₆ᵢʲ/r₆ + s₈ C₈ᵢʲ/r₈  (with damping function)")
print()
print("Usage in PySCF:")
print("  mf = dft.RKS(mol)")
print("  mf.xc = 'B3LYP'")
print("  from pyscf import dftd3  # requires dftd3 Python bindings")
print("  mf = dftd3.dftd3(mf)    # wraps mf with D3 correction")
print()
print("Usage in ORCA:")
print("  ! B3LYP-D3BJ def2-SVP Opt")
print("  ! PBE0-D3/def2-TZVP Energy")
print()
print("Common dispersion variants:")
print("  -D3(BJ): Becke-Johnson damping (recommended for most systems)")
print("  -D3(0):  Zero-damping (Grimme 2010, original)")
print("  -D4:     Charge-dependent, more accurate for ionic systems")

## 2. DFT Functional Reference Table

| Functional | Type | %HF Exchange | Dispersion | Cost | Accuracy | Good For | Caution |
|------------|------|:---:|:---:|:---:|:---:|---------|----------|
| SVWN | LDA | 0 | ✗ | ★ | ★★ | Solids, quick tests | Overbinds molecules |
| PBE | GGA | 0 | ✗ | ★★ | ★★★ | Solids, surfaces | Self-interaction |
| BLYP | GGA | 0 | ✗ | ★★ | ★★★ | Organics | Dispersion missing |
| TPSS | meta-GGA | 0 | ✗ | ★★★ | ★★★ | General chemistry | Slow |
| M06-L | meta-GGA | 0 | ✗ | ★★★ | ★★★★ | Transition metals | Sensitive to grid |
| B3LYP | Hybrid | 20 | ✗ | ★★★ | ★★★★ | Organics, pharmacy | No dispersion; fails TM |
| B3LYP-D3 | Hybrid+D | 20 | ✓ | ★★★ | ★★★★ | General organics | TM still tricky |
| PBE0 | Hybrid | 25 | ✗ | ★★★ | ★★★★ | General chemistry | Dispersion missing |
| PBE0-D3 | Hybrid+D | 25 | ✓ | ★★★ | ★★★★★ | **Recommended general** | |
| TPSSh | Hybrid | 10 | ✗ | ★★★ | ★★★★ | Transition metals | |
| M06 | Hybrid | 27 | ✗ | ★★★★ | ★★★★ | Main group + TM | Sensitive to grid |
| M06-2X | Hybrid | 54 | ✗ | ★★★★ | ★★★★★ | Thermochem, kinetics | Fails for TM |
| ωB97X-D | Range-sep | var | ✓ | ★★★★ | ★★★★★ | CT excitations, NCI | Expensive |
| CAM-B3LYP | Range-sep | var | ✗ | ★★★★ | ★★★★ | CT, long-range | |
| B2-PLYP | Double hybrid | 53 | ✗ | ★★★★★ | ★★★★★ | High accuracy | Very expensive |

**Course recommendations:**
- **Geometry optimization**: `B3LYP/def2-SVP` or `PBE0/def2-SVP`
- **Energies**: `PBE0-D3/def2-TZVP` or `B3LYP-D3BJ/def2-TZVP`
- **Transition metals**: `TPSSh/def2-SVP` or `M06/def2-SVP`
- **Excited states**: `CAM-B3LYP/def2-SVP` (TD-DFT)

## 🔬 Research Connection

DFT is the workhorse of modern computational chemistry and materials science:

- **~50% of all computational chemistry papers** use DFT (B3LYP alone has >100,000 citations)
- **Drug design**: DFT computes binding site interactions, reaction mechanisms in metalloenzymes
- **Heterogeneous catalysis**: PBE/plane-waves for surface reactions (e.g., CO₂ reduction)
- **Battery materials**: DFT predicts lithium intercalation energies in electrode materials
- **Nobel Prizes**: Kohn (1998 Chemistry), Karplus/Levitt/Warshel (2013 Chemistry)

**The DFT accuracy crisis**: B3LYP was found to fail for dispersion-dominated systems,
kinetics (reaction barriers), and transition metal complexes. This drove development
of M06-2X, ωB97X-D, and dispersion corrections, now standard in publications.

**Key reference:** Becke, A.D. (1993). *J. Chem. Phys.* **98**, 5648 — the B3LYP paper,
one of the most cited scientific papers of all time (~90,000 citations).

## 📋 Summary

| Concept | Key Points |
|---------|----------|
| HK Theorem 1 | Ground state energy is a unique functional of $n_0(\mathbf{r})$ |
| HK Theorem 2 | Variational principle: true $n_0$ minimizes $E[n]$ |
| Kohn-Sham DFT | Non-interacting reference with same density; same structure as HF |
| LDA | Uses only $n(\mathbf{r})$; overbinds; good for solids |
| GGA | Adds $\nabla n$; better for molecules; missing dispersion |
| Hybrid | Mixes HF exchange; best for most molecular applications |
| Dispersion | -D3(BJ) adds $C_6/r^6$ correction; always use for weak interactions |

**When DFT works well:** Geometries (±0.01-0.02 Å), reaction energies (±2 kcal/mol),
vibrational frequencies, dipole moments, NMR shifts

**When DFT struggles:** Strong correlation (multi-reference), charge-transfer excitations (without range-separation), dispersion (without correction), delocalization error

## 📝 Exercises

1. **Jacob's Ladder**: For each rung (LDA, GGA, meta-GGA, hybrid), name one functional
   not listed in the notebook. Look up its year of publication and the journal it appeared in.

2. **Functional comparison**: Run the water functional comparison with `aug-cc-pVTZ`
   instead of `def2-SVP`. Does the ordering of energies change? Why might it?

3. **Dispersion in benzene dimer**: Using PySCF, compute the interaction energy of
   two benzene molecules (face-to-face at 3.8 Å) with B3LYP/6-31G* (no dispersion).
   What sign is the interaction? Is this physical?

4. **Exact exchange fraction**: B3LYP uses 20% HF exchange. Try changing this using
   `mf.xc = '0.15*HF + 0.85*B88, LYP'` and `mf.xc = '0.30*HF + 0.70*B88, LYP'`.
   How do the energy and dipole moment of water change?

5. **DFT vs HF correlation energy**: Compare the energy of H₂O computed by
   RHF/def2-SVP and PBE/def2-SVP. The difference approximates how much correlation
   energy DFT captures. How does this compare to the MP2 correlation energy?